# R8 — Lakebase + external join (provider directory)

Hospital accreditation and contact details move out of spreadsheets and into a real **transactional system** — Lakebase Postgres — then join against the hospital dimension so analytics can see, per hospital, how many claims were admitted **after its accreditation lapsed**. A live connection into a transactional store behaves differently from an ordinary lakehouse table, and that difference is respected, not papered over with a view that only works some of the time.

**Status: MAIN PATH COMPLETE — Lakebase.** The provider directory Postgres database was imported and registered as the Unity Catalog catalog **`claims_lakebase`**; the accreditation table is `claims_lakebase.public.provider_directory` (a FOREIGN table over Lakebase, queryable directly from the notebook session — the earlier "SQL warehouse only" assumption did not hold). The join cell below reads the live Lakebase source and writes `claims.analytics.r8_provider_join`. Evidence: `evidence/r8_source.md` records `source: lakebase`; the earlier CSV-fallback build (same numbers, verified) is superseded.

**Lakebase main path — as executed:**
1. Lakebase Postgres project + branch created; database `providers` loaded with `provider_directory.sql`.
2. The loaded provider directory registered as Unity Catalog catalog `claims_lakebase` (visible in the catalog list; table `public.provider_directory` is FOREIGN).
3. Join cell below builds `claims.analytics.r8_provider_join` from `claims_lakebase.public.provider_directory`.

**Fallback 1 — Lakehouse Federation** (only if Lakebase were unavailable): create a foreign catalog over the Postgres source and run the same join cell; record `source: federation`.

**Fallback 2 — CSV** (superseded, kept for history): load the CSV twin into `claims.raw.provider_directory` and record `source: csv`. The CSV twin remains at `claims.raw.provider_directory` and `/Volumes/claims/raw/landing/provider_directory.csv`; its verified results (HOS-024 43, HOS-035 34, HOS-009 33, HOS-005 8 — total 118) matched the Lakebase dry-run exactly.
![claims_lakebase catalog — Lakebase Postgres autoscaling, project claimslakebase, branch production, public schema with provider_directory](evidence/screenshots/r8_lakebase_catalog.png)


In [0]:
%sql
-- r8_provider_join: per hospital, accreditation + claims admitted AFTER its accreditation lapsed.
-- Source: Lakebase — Unity Catalog catalog `claims_lakebase` (registered from the loaded provider
-- directory Postgres database). Accreditation table: claims_lakebase.public.provider_directory.
CREATE OR REPLACE TABLE claims.analytics.r8_provider_join AS
WITH providers AS (
  SELECT * FROM claims_lakebase.public.provider_directory
)
SELECT
  h.hospital_id,
  h.hospital_name,
  h.city_tier,
  pr.accreditation,
  pr.accredited_until,                                          -- DATE; NULL = accreditation still open
  CAST(COALESCE(SUM(
    CASE
      WHEN pr.accredited_until IS NOT NULL
       AND to_date(CAST(f.admission_date_key AS STRING), 'yyyyMMdd') > pr.accredited_until
      THEN 1 ELSE 0
    END
  ), 0) AS INT) AS claims_after_lapse
FROM claims.analytics.dim_hospital h
LEFT JOIN providers pr USING (hospital_id)
LEFT JOIN claims.analytics.fact_claims f ON f.hospital_sk = h.hospital_sk
GROUP BY h.hospital_id, h.hospital_name, h.city_tier, pr.accreditation, pr.accredited_until
ORDER BY h.hospital_id;

In [0]:
%sql
SELECT * FROM claims.analytics.r8_provider_join ORDER BY claims_after_lapse DESC;